In [6]:
import onnx

import os
from pathlib import Path

import torch

from src.CNN_model_definitions import create_tiny_model
from src.ViT_model_definitions import SmallViT
from src.hybrid_model_definition import create_hybrid_model

### 01. ONNX from the champion CNN model

In [8]:
models_dir = Path("./models")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used is {device}")

dummy_input = torch.randn(
    1, 3, 64, 64,
    device=device
)

Device used is cuda


In [3]:
cnn_champion_path = models_dir / "cnn_0_best.pth"

model = create_tiny_model()
model.to(device)
model.load_state_dict(
        torch.load(cnn_champion_path, map_location=device)
    )

<All keys matched successfully>

In [4]:
onnx_path = models_dir / "PyTorch_CNN.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=17
)

print(f"ONNX model saved to: {onnx_path}")

/tmp/ipykernel_3777/3825990806.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


ONNX model saved to: models/PyTorch_CNN.onnx


Unfortunately using dynamo=True causes libraries version mismatches for now.

In [10]:
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print("ONNX model is valid.")

ONNX model is valid.


### 02. ONNX from the ViT model

In [6]:
vit_model = SmallViT()

vit_model_path = models_dir) / "vit_pure.pth"

vit_model.load_state_dict(
    torch.load(vit_model_path, weights_only=True)
)

vit_model.to(device)

SmallViT(
  (patch_embedding): PatchEmbedding(
    (projection): Linear(in_features=192, out_features=64, bias=True)
  )
  (transformer_blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (dropout1): Dropout(p=0.1, inplace=False)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=64, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=64, out_features=64, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

In [7]:
onnx_vit_path = models_dir / "PyTorch_ViT.onnx"

In [8]:
torch.onnx.export(
    vit_model,
    dummy_input,
    onnx_vit_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=17
)

print(f"ONNX model saved to: {onnx_vit_path}")

/tmp/ipykernel_3777/2736709683.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


ONNX model saved to: models/PyTorch_ViT.onnx


In [9]:
onnx_vit_model = onnx.load(onnx_vit_path)
onnx.checker.check_model(onnx_vit_model)

print("ONNX ViT model is valid.")

ONNX ViT model is valid.


### 03. ONNX from the hybrid model

In [9]:
hybrid_model = create_hybrid_model().to(device)

hybrid_model_path = models_dir / "hybrid.pth"
onnx_hybrid_path = models_dir / "PyTorch_hybrid.onnx"

hybrid_model.load_state_dict(
    torch.load(hybrid_model_path, weights_only=True)
)


<All keys matched successfully>

In [10]:
torch.onnx.export(
    hybrid_model,
    dummy_input,
    onnx_hybrid_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=17
)

print(f"ONNX model saved to: {onnx_hybrid_path}")

/tmp/ipykernel_80/1596881693.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


ONNX model saved to: models/PyTorch_ViT.onnx
